# FIFA World Cup 2026 — Step 3: XGBoost Model Training & Prediction

Train an XGBoost classifier on historical match features and predict 2026 outcomes.

**Workflow:**
  1. Load features_train.csv (2000–2021 matches)
  2. Load features_test.csv (2022 World Cup test set)
  3. Train XGBoost classifier
  4. Evaluate on test set (accuracy, precision, recall)
  5. Load features_2026.csv
  6. Predict all 2026 group stage matches
  7. Simulate group standings & qualify teams

In [1]:
import sys
!{sys.executable} -m pip install xgboost


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys

!{sys.executable} -m pip install numpy pandas scikit-learn xgboost


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import pandas as pd
import numpy as np
import os
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "data"
print(f" Data directory: {DATA_DIR}")

 Data directory: data


## 1. Load Training & Test Data

In [24]:
def load_features():
    print("\n[1/5] Loading feature sets...")
    
    train = pd.read_csv(os.path.join(DATA_DIR, "features_train.csv"))
    test = pd.read_csv(os.path.join(DATA_DIR, "features_test.csv"))
    fixtures_2026 = pd.read_csv(os.path.join(DATA_DIR, "features_2026.csv"))
    
    print(f"    ✓ Training set:  {len(train):,} matches (2000–2021)")
    print(f"    ✓ Test set:      {len(test):,} matches (2022 World Cup)")
    print(f"    ✓ 2026 fixtures: {len(fixtures_2026)} group-stage matches")
    
    return train, test, fixtures_2026

train, test, fixtures_2026 = load_features()


[1/5] Loading feature sets...
    ✓ Training set:  20,775 matches (2000–2021)
    ✓ Test set:      969 matches (2022 World Cup)
    ✓ 2026 fixtures: 72 group-stage matches


In [25]:
FEATURE_COLS = [
    "home_elo", "away_elo", "elo_diff", "elo_win_prob",
    "home_form_win", "home_form_draw",
    "away_form_win", "away_form_draw",
    "form_diff",
    "home_avg_scored", "home_avg_conceded",
    "away_avg_scored", "away_avg_conceded",
    "xgd",
    "h2h_home_win_rate", "h2h_draw_rate",
    "tournament_weight", "is_neutral", "is_worldcup",
]

result_map = {
    "L": 0,
    "D": 1,
    "W": 2
}

train["target"] = train["result"].map(result_map)
test["target"] = test["result"].map(result_map)

TARGET_COL = "target"  # W=2, D=1, L=0

# Prepare train set
X_train = train[FEATURE_COLS].copy()
y_train = train[TARGET_COL].copy()

# Prepare test set
X_test = test[FEATURE_COLS].copy()
y_test = test[TARGET_COL].copy()

print(f"\nFeature matrix shape: {X_train.shape}")
print(f"Target distribution (train):  {dict(y_train.value_counts().sort_index())}")
print(f"Target distribution (test):   {dict(y_test.value_counts().sort_index())}")


Feature matrix shape: (20775, 19)
Target distribution (train):  {0: np.int64(5900), 1: np.int64(4856), 2: np.int64(10019)}
Target distribution (test):   {0: np.int64(267), 1: np.int64(220), 2: np.int64(482)}


## 3. Train XGBoost Classifier

In [6]:
import sys

!{sys.executable} -m pip uninstall -y xgboost
!{sys.executable} -m pip install --upgrade xgboost

Found existing installation: xgboost 3.2.0
Uninstalling xgboost-3.2.0:
  Successfully uninstalled xgboost-3.2.0


You can safely remove it manually.


  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import sklearn
import xgboost as xgb

print(sklearn.__version__)
print(xgb.__version__)

1.5.2
3.2.0


In [8]:
import sys
!{sys.executable} -m pip install scikit-learn==1.5.2 --force-reinstall

  Using cached scikit_learn-1.5.2-cp311-cp311-win_amd64.whl.metadata (13 kB)
  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.5.2-cp311-cp311-win_amd64.whl (11.0 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl (12.6 MB)
Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl (36.6 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

  Attempting uninstall: threadpoolctl

    Found existing installation: threadpoolctl 3.6.0

    Uninstalling threadpoolctl-3.6.0:

   ---------------------------------------- 0/5 [threadpoolctl]
   ---------------------------------------- 0/5 [threadpoolctl]
   ---------------------------------------- 0/5 [threadpoolctl]
   --------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.21 requires numpy<2, but you have numpy 2.4.6 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import sklearn
import xgboost as xgb

print(sklearn.__version__)
print(xgb.__version__)

model = xgb.XGBClassifier()

1.5.2
3.2.0


In [10]:

print("\n[2/5] Training XGBoost model...")

# Initialize XGBoost classifier for multi-class prediction (L/D/W)
model = xgb.XGBClassifier(
    n_estimators=400,          # number of boosting rounds
    max_depth=6,               # tree depth
    learning_rate=0.1,         # shrinkage
    subsample=0.8,             # fraction of samples per tree
    colsample_bytree=0.8,      # fraction of features per tree
    objective="multi:softprob",
    early_stopping_rounds=20,# multiclass classification
    random_state=42,
    n_jobs=-1,                 # use all cores
)

# Train model
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print("     Model training complete")


[2/5] Training XGBoost model...
     Model training complete


## 4. Evaluate on Test Set (2022 World Cup)

In [11]:
print("\n[3/5] Evaluating model on 2022 World Cup test set...")

# Make predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)  # probability for each class

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"\n    Accuracy: {accuracy:.2%}")

# Detailed metrics per class
print(f"\n    Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Loss", "Draw", "Win"]))

# Feature importance
importance_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(f"\n    Top 10 Most Important Features:")
for idx, row in importance_df.head(10).iterrows():
    print(f"      {row['feature']:<25} {row['importance']:.4f}")


[3/5] Evaluating model on 2022 World Cup test set...

    Accuracy: 63.78%

    Classification Report:
              precision    recall  f1-score   support

        Loss       0.57      0.75      0.65       267
        Draw       0.35      0.06      0.10       220
         Win       0.70      0.84      0.76       482

    accuracy                           0.64       969
   macro avg       0.54      0.55      0.50       969
weighted avg       0.58      0.64      0.58       969


    Top 10 Most Important Features:
      elo_diff                  0.3279
      elo_win_prob              0.1423
      is_neutral                0.0590
      form_diff                 0.0475
      away_elo                  0.0361
      is_worldcup               0.0328
      away_form_win             0.0326
      h2h_home_win_rate         0.0317
      home_elo                  0.0307
      tournament_weight         0.0291


## 5. Predict 2026 World Cup Fixtures

In [27]:
print("\n[4/5] Predicting 2026 World Cup fixtures...")

# Prepare 2026 fixtures
X_2026 = fixtures_2026[FEATURE_COLS].copy()

# Make predictions
pred_2026 = model.predict(X_2026)
proba_2026 = model.predict_proba(X_2026)

# Add predictions to fixtures dataframe
fixtures_2026["predicted_outcome"] = pred_2026
fixtures_2026["prob_loss"] = proba_2026[:, 0]
fixtures_2026["prob_draw"] = proba_2026[:, 1]
fixtures_2026["prob_win"]  = proba_2026[:, 2]

# Map outcome to labels
outcome_map = {0: "L", 1: "D", 2: "W"}
fixtures_2026["outcome_label"] = fixtures_2026["predicted_outcome"].map(outcome_map)

print(f"    ✓ Predictions generated for {len(fixtures_2026)} matches")


[4/5] Predicting 2026 World Cup fixtures...
    ✓ Predictions generated for 72 matches


In [28]:
print("\n[5/5] 2026 FIFA World Cup Predictions by Group\n")
print("=" * 90)

for group in sorted(fixtures_2026["group"].unique()):
    group_matches = fixtures_2026[fixtures_2026["group"] == group].sort_values("home_team")
    print(f"\n  GROUP {group}:")
    for _, match in group_matches.iterrows():
        home = match["home_team"]
        away = match["away_team"]
        outcome = match["outcome_label"]
        prob_home = match["prob_win"] * 100
        prob_draw = match["prob_draw"] * 100
        prob_away = match["prob_loss"] * 100
        
        print(f"    {home:<15} vs {away:<15} → {outcome} | Home: {prob_home:5.1f}%  Draw: {prob_draw:5.1f}%  Away: {prob_away:5.1f}%")

print(f"\n" + "=" * 90)


[5/5] 2026 FIFA World Cup Predictions by Group


  GROUP L:
    Argentina       vs Chile           → W | Home:  72.5%  Draw:  12.6%  Away:  14.9%
    Argentina       vs Peru            → W | Home:  67.4%  Draw:  14.7%  Away:  17.9%
    Argentina       vs Australia       → W | Home:  68.9%  Draw:  11.4%  Away:  19.6%
    Austria         vs Ukraine         → W | Home:  67.0%  Draw:  13.8%  Away:  19.1%
    Austria         vs Nigeria         → W | Home:  76.9%  Draw:  11.0%  Away:  12.1%
    Belgium         vs Croatia         → L | Home:  17.1%  Draw:  11.8%  Away:  71.1%
    Belgium         vs Thailand        → W | Home:  81.3%  Draw:   6.7%  Away:  12.0%
    Brazil          vs Cameroon        → W | Home:  76.6%  Draw:  10.9%  Away:  12.6%
    Brazil          vs Ecuador         → W | Home:  60.5%  Draw:  13.7%  Away:  25.8%
    Brazil          vs Colombia        → W | Home:  59.2%  Draw:  15.9%  Away:  24.9%
    Canada          vs Slovakia        → W | Home:  76.9%  Draw:  11.8%  Away: 

## 7. Simulate Group Standings

In [29]:
def simulate_group_standings(group_matches):
    """Simulate group standings based on predicted outcomes."""
    
    teams = pd.concat([
        group_matches["home_team"],
        group_matches["away_team"]
    ]).unique()
    
    standings = pd.DataFrame({
        "team": teams,
        "played": 0,
        "wins": 0,
        "draws": 0,
        "losses": 0,
        "points": 0
    })
    
    for _, match in group_matches.iterrows():
        home = match["home_team"]
        away = match["away_team"]
        outcome = match["predicted_outcome"]  # 0=L, 1=D, 2=W
        
        # Update played
        standings.loc[standings["team"] == home, "played"] += 1
        standings.loc[standings["team"] == away, "played"] += 1
        
        # Update results
        if outcome == 2:  # Home win
            standings.loc[standings["team"] == home, "wins"] += 1
            standings.loc[standings["team"] == away, "losses"] += 1
            standings.loc[standings["team"] == home, "points"] += 3
        elif outcome == 1:  # Draw
            standings.loc[standings["team"] == home, "draws"] += 1
            standings.loc[standings["team"] == away, "draws"] += 1
            standings.loc[standings["team"] == home, "points"] += 1
            standings.loc[standings["team"] == away, "points"] += 1
        else:  # Away win
            standings.loc[standings["team"] == home, "losses"] += 1
            standings.loc[standings["team"] == away, "wins"] += 1
            standings.loc[standings["team"] == away, "points"] += 3
    
    # Sort by points descending
    standings = standings.sort_values(
        by="points",
        ascending=False
    ).reset_index(drop=True)
    
    return standings


print("\n" + "=" * 90)
print("  PREDICTED GROUP STANDINGS")
print("=" * 90)

for group in sorted(fixtures_2026["group"].unique()):
    group_matches = fixtures_2026[fixtures_2026["group"] == group]
    standings = simulate_group_standings(group_matches)
    
    print(f"\n  GROUP {group}:")
    print(f"  {'':<4}  {'Team':<20}  P   W   D   L   Pts")
    print(f"  {'':<4}  {'-' * 20}  -   -   -   -   ---")
    
    for idx, (_, row) in enumerate(standings.iterrows(), 1):
        qualifier = "✓ Q" if idx <= 2 else "  "
        print(f"  {qualifier}  {row['team']:<20}  {int(row['played'])}   {int(row['wins'])}   {int(row['draws'])}   {int(row['losses'])}   {int(row['points']):3d}")

print(f"\n" + "=" * 90)


  PREDICTED GROUP STANDINGS

  GROUP L:
        Team                  P   W   D   L   Pts
        --------------------  -   -   -   -   ---
  ✓ Q  Argentina             3   3   0   0     9
  ✓ Q  Mexico                3   3   0   0     9
      Spain                 3   3   0   0     9
      France                3   3   0   0     9
      England               3   3   0   0     9
      Canada                3   3   0   0     9
      Japan                 3   3   0   0     9
      Brazil                3   3   0   0     9
      Italy                 3   3   0   0     9
      Morocco               3   3   0   0     9
      Austria               3   3   0   0     9
      South Korea           3   3   0   0     9
      Croatia               3   2   0   1     6
      Venezuela             3   2   0   1     6
      Panama                3   2   0   1     6
      Portugal              3   2   0   1     6
      Iran                  3   2   0   1     6
      Sweden                3   2   0   1

In [30]:
# Save full predictions
pred_cols = [
    "group", "home_team", "away_team", "outcome_label",
    "prob_win", "prob_draw", "prob_loss", "elo_diff", "home_elo", "away_elo"
]
predictions_output = fixtures_2026[pred_cols].copy()
predictions_output.to_csv(os.path.join(DATA_DIR, "2026_predictions.csv"), index=False)
print(f"\n✅ Predictions saved → data/2026_predictions.csv")

# Model performance summary
print(f"\n" + "=" * 90)
print(f"  MODEL SUMMARY")
print(f"=" * 90)
print(f"  Algorithm:          XGBoost (Gradient Boosting)")
print(f"  Training samples:   {len(train):,}")
print(f"  Test accuracy:      {accuracy:.2%}")
print(f"  Feature count:      {len(FEATURE_COLS)}")
top_3 = importance_df.head(3)["feature"].tolist()
print(f"  Top 3 features:     {', '.join(top_3)}")
print(f"=" * 90 + "\n")


✅ Predictions saved → data/2026_predictions.csv

  MODEL SUMMARY
  Algorithm:          XGBoost (Gradient Boosting)
  Training samples:   20,775
  Test accuracy:      63.78%
  Feature count:      19
  Top 3 features:     elo_diff, elo_win_prob, is_neutral

